In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import snntorch as snn
from snntorch import surrogate

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from pathlib import Path
from tqdm import tqdm # Dodano import tqdm
from collections import Counter
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix, top_k_accuracy_score, accuracy_score, precision_recall_fscore_support
import h5py # Nadal importowane, choć dla BIN nieużywane w datasecie
import math
from scipy.stats import pearsonr # For correlation

# --- FUNKCJE I KLASY DLA DANYCH RZECZYWISTYCH (.BIN) ---
def load_bin_events(path):
  raw = np.fromfile(path, dtype=np.uint8)
  raw = raw[:len(raw) // 5 * 5]
  x = raw[0::5].astype(np.int32)
  y = raw[1::5].astype(np.int32)
  p = (raw[2::5] >> 7) & 1
  p = np.where(p == 1, 1, -1)

  t_high = (raw[2::5] & 0x7F).astype(np.uint32)
  t_mid  = raw[3::5].astype(np.uint32)
  t_low  = raw[4::5].astype(np.uint32)
  t = (t_high << 16) | (t_mid << 8) | t_low

  return x, y, t.astype(np.float32), p

class EventDatasetBIN(Dataset):
  def __init__(self, root_dir, classes, T=10, H=160, W=160, clip_value=5.0, augment=False): # Zmieniono domyślne H, W
    self.root_dir = Path(root_dir)
    self.classes = classes
    self.class_to_idx = {c: i for i, c in enumerate(classes)}
    self.T, self.H, self.W = T, H, W
    self.clip_value = clip_value
    self.augment = augment

    self.samples = []
    if not self.root_dir.exists():
        print(f"Warning: Dataset root directory does not exist: {self.root_dir}")
        return

    for cls in classes:
        class_path = self.root_dir / cls
        if not class_path.exists():
            print(f"Ostrzeżenie: Katalog klasy {class_path} nie istnieje. Pomijam.")
            continue
        for f in class_path.glob("*.bin"):
            self.samples.append((f, self.class_to_idx[cls]))
    
    if not self.samples:
        print(f"Warning: No event files found in {self.root_dir} for given classes (looking for .bin).")

  def __len__(self):
      return len(self.samples)

  def __getitem__(self, idx):
      path, label = self.samples[idx]
      x, y, t, p = load_bin_events(path)

      if len(t) == 0:
          return torch.zeros((2, self.H, self.W, self.T), dtype=torch.float32), torch.tensor(label, dtype=torch.long)

      if self.augment and np.random.rand() > 0.5:
          x = (self.W - 1) - x

      t_min = t.min()
      t_max = t.max()
      if (t_max - t_min) < 1e-9:
          t_normalized = np.zeros_like(t)
      else:
          t_normalized = (t - t_min) / (t_max - t_min)
          
      voxels = np.zeros((2, self.H, self.W, self.T), dtype=np.float32)
      time_bins = (t_normalized * (self.T - 1)).astype(np.int32)

      for xi, yi, pi, bi in zip(x, y, p, time_bins):
          if 0 <= xi < self.W and 0 <= yi < self.H:
              ch = 0 if pi > 0 else 1
              voxels[ch, yi, xi, bi] += 1.0

      voxels = np.clip(voxels, 0.0, self.clip_value)
      voxels /= self.clip_value

      voxels *= 1.5

      return torch.from_numpy(voxels), torch.tensor(label, dtype=torch.long)

# --- KONIEC FUNKCJI I KLAS DLA DANYCH RZECZYWISTYCH ---


# --- Definicje dla ResNet SNN ---
class LIFNode(nn.Module):
    def __init__(self, thresh=1.0, tau=0.9, gamma=1.0, **kwargs): # Dodano gamma
        super().__init__()
        self.thresh = thresh
        self.tau = tau
        self.gamma = gamma # Przechowywanie gamma
        self.mem = None
        self.spike = None
        # surrogate.fast_sigmoid() używa gamma jako parametru 'slope'
        self.spike_fn = snn.Leaky(beta=self.tau, threshold=self.thresh, spike_grad=surrogate.fast_sigmoid(slope=self.gamma)) 

        self.total_spikes = 0
        self.num_neurons = 0
        self.num_time_steps = 0
        self.batch_size = 0

    def forward(self, x):
        B, T, C, H, W = x.shape

        self.total_spikes = 0
        self.num_neurons = x.shape[2:].numel()
        self.num_time_steps = T
        self.batch_size = B

        spikes_output = []
        mem_curr = self.spike_fn.init_leaky().to(x.device)
        for t in range(x.shape[1]):
            spike_t, mem_curr = self.spike_fn(x[:, t, ...], mem_curr)
            spikes_output.append(spike_t)
            self.total_spikes += spike_t.sum().item()
        return torch.stack(spikes_output, dim=1)

    def get_avg_firing_rate(self):
        if self.batch_size * self.num_time_steps * self.num_neurons == 0:
            return 0.0
        return self.total_spikes / (self.batch_size * self.num_time_steps * self.num_neurons)


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1, dropout_p=0.3, **lif_parameters):
        super().__init__()
        filtered_lif_parameters = {
            k: v for k, v in lif_parameters.items() if k in ['thresh', 'tau', 'gamma']
        }

        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.spike1 = LIFNode(**filtered_lif_parameters)
        self.dropout1 = nn.Dropout2d(p=dropout_p)

        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.spike2 = LIFNode(**filtered_lif_parameters)
        self.dropout2 = nn.Dropout2d(p=dropout_p)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion * planes)
            )

    def forward(self, x):
        B, T, C, H, W = x.shape

        x_flat_for_conv_bn = x.contiguous().view(B*T, C, H, W)

        out = self.conv1(x_flat_for_conv_bn)
        out = self.bn1(out)
        out = out.view(B, T, out.shape[1], out.shape[2], out.shape[3])
        out = self.spike1(out)
        out = self.dropout1(out.contiguous().view(B*T, *out.shape[2:])).view(B, T, *out.shape[2:])

        out = self.conv2(out.contiguous().view(B*T, *out.shape[2:]))
        out = self.bn2(out)
        out = out.view(B, T, out.shape[1], out.shape[2], out.shape[3])
        out = self.spike2(out)
        out = self.dropout2(out.contiguous().view(B*T, *out.shape[2:])).view(B, T, *out.shape[2:])

        shortcut_out = self.shortcut(x_flat_for_conv_bn).view(B, T, -1, *out.shape[3:]) if len(self.shortcut) > 0 else x

        out += shortcut_out
        return out

class ResNet_SNN(nn.Module):
    def __init__(self, block, num_blocks, num_classes=101, in_channels=2, dropout_p=0.3, **lif_parameters):
        super().__init__()
        self.in_planes = 64
        self.dropout_p = dropout_p
        self.T = lif_parameters.get('T', 10)

        filtered_lif_parameters_for_lif_node = {
            k: v for k, v in lif_parameters.items() if k in ['thresh', 'tau', 'gamma']
        }

        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.spike1 = LIFNode(**filtered_lif_parameters_for_lif_node)
        self.dropout_initial = nn.Dropout2d(p=self.dropout_p)

        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1, dropout_p=self.dropout_p, **filtered_lif_parameters_for_lif_node)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2, dropout_p=self.dropout_p, **filtered_lif_parameters_for_lif_node)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2, dropout_p=self.dropout_p, **filtered_lif_parameters_for_lif_node)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2, dropout_p=self.dropout_p, **filtered_lif_parameters_for_lif_node)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.flatten = nn.Flatten()
        self.linear = nn.Linear(512 * block.expansion, num_classes)

        self.lif_modules = nn.ModuleList([self.spike1])
        for layer in [self.layer1, self.layer2, self.layer3, self.layer4]:
            for block_module in layer.children():
                if isinstance(block_module, BasicBlock):
                    self.lif_modules.append(block_module.spike1)
                    self.lif_modules.append(block_module.spike2)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)


    def _make_layer(self, block, planes, num_blocks, stride, dropout_p, **lif_parameters):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for stride_val in strides:
            layers.append(block(self.in_planes, planes, stride_val, dropout_p=dropout_p, **lif_parameters))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        B, T, C, H, W = x.shape

        x_flat_for_conv_bn = x.contiguous().view(B*T, C, H, W)

        out = self.conv1(x_flat_for_conv_bn)
        out = self.bn1(out)
        out = out.view(B, T, out.shape[1], out.shape[2], out.shape[3])
        out = self.spike1(out)
        out = self.dropout_initial(out.contiguous().view(B*T, *out.shape[2:])).view(B, T, *out.shape[2:])

        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)

        out_flat = out.contiguous().view(B*T, *out.shape[2:])
        out_pooled = self.avgpool(out_flat)
        out_final = self.flatten(out_pooled)
        out_final = self.linear(out_final)

        out_reshaped = out_final.view(B, T, -1)
        return out_reshaped


# Factory functions
def resnet18(**kwargs):
    return ResNet_SNN(BasicBlock, [2, 2, 2, 2], **kwargs)

def resnet34(**kwargs):
    return ResNet_SNN(BasicBlock, [3, 4, 6, 3], **kwargs)

# --- Koniec definicji ResNet SNN ---


def evaluate(model, loader, device, criterion, classes):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    all_labels = []
    all_probs = []

    # Dodano pasek postępu do ewaluacji
    with torch.no_grad():
        for x, y in tqdm(loader, desc="Ewaluacja modelu"): # Dodano tqdm
            x, y = x.to(device), y.to(device)

            x = x.permute(0, 4, 1, 2, 3)

            out_seq = model(x)
            out = out_seq.sum(1)

            loss = criterion(out, y)
            total_loss += loss.item()

            probs = F.softmax(out, dim=1)
            preds = probs.argmax(dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

            all_labels.extend(y.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    val_loss = total_loss / len(loader)
    val_acc = correct / total

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)

    top1 = top_k_accuracy_score(all_labels, all_probs, k=1)
    top5 = top_k_accuracy_score(all_labels, all_probs, k=min(5, len(classes))) # Poprawka literówki

    return val_loss, val_acc, top1, top5, all_labels, all_probs, 0.0

# --- Konfiguracja dla wizualizacji ---
# --- ZMIANA: Ponowne włączenie detekcji GPU ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Używane urządzenie: {device}")

# --- ZMIANA: Ponowne włączenie ustawień CUDA backend ---
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

# Ścieżki bazowe dla wyników i wykresów
BASE_SNN_DIR = Path(r"C:\Users\Kacper\OneDrive\Desktop\SNN")
EXPERIMENT_NAME = "results-real_ResNet" # Zmieniona nazwa eksperymentu na ResNet18 dla danych syntetycznych

RESULTS_DIR = BASE_SNN_DIR / "results" / EXPERIMENT_NAME
PLOTS_DIR = BASE_SNN_DIR / "plots" / EXPERIMENT_NAME
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = RESULTS_DIR / "checkpoint.pth"
BEST_MODEL_PATH = RESULTS_DIR / "best_model.pth"

# Ścieżki do danych syntetycznych H5
TRAIN_ROOT = BASE_SNN_DIR / "data" / "N-Caltech101-full-split" / "train"
TEST_ROOT  = BASE_SNN_DIR / "data" / "N-Caltech101-full-split" / "test"

# Wczytanie nazw klas
CLASSES = []
NUM_CLASSES = 0
try:
    if TRAIN_ROOT.exists():
        CLASSES = sorted([p.name for p in TRAIN_ROOT.iterdir() if p.is_dir()])
        try:
            CLASSES = sorted(CLASSES, key=int)
        except ValueError:
            pass
        NUM_CLASSES = len(CLASSES)
        if NUM_CLASSES == 0:
            print(f"Ostrzeżenie: Nie znaleziono katalogów klas w '{TRAIN_ROOT}'. Użycie domyślnych 101 klas.")
            NUM_CLASSES = 101
            CLASSES = [f"{i}" for i in range(NUM_CLASSES)]
    else:
        print(f"Ostrzeżenie: Katalog danych '{TRAIN_ROOT}' nie istnieje. Użycie domyślnych 101 klas.")
        NUM_CLASSES = 101
        CLASSES = [f"Class_{i}" for i in range(NUM_CLASSES)]
except Exception as e:
    print(f"Błąd podczas wczytywania klas: {e}. Użycie domyślnych 101 klas.")
    NUM_CLASSES = 101
    CLASSES = [f"Class_{i}" for i in range(NUM_CLASSES)]

print(f"Wykryta liczba klas: {NUM_CLASSES}")

# Hiperparametry modelu ResNet18 SNN
DATASET_T = 10 # Zgodnie z twoim kodem treningowym dla syntetycznych
MODEL_PARAMS = {
    "num_classes": NUM_CLASSES,
    "in_channels": 2,
    "thresh": 0.7,
    "tau": 0.95,
    "gamma": 1.0,
    "dropout_p": 0.1,
    "T": DATASET_T
}

# --- Wczytanie historii z checkpointu ---
history = None
if CHECKPOINT_PATH.exists():
    print(f"Wczytywanie checkpointu z {CHECKPOINT_PATH}...")
    try:
        # --- ZMIANA: load checkpoint na odpowiednie urządzenie ---
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
        history = checkpoint["history"]
        print("Historia wczytana pomyślnie.")
    except Exception as e:
        print(f"Błąd podczas wczytywania checkpointu: {e}")
        history = None
else:
    print(f"Błąd: Plik checkpointu nie znaleziony pod {CHECKPOINT_PATH}.")

# Robustne przetwarzanie historii
if history:
    expected_keys = ["epoch", "train_loss", "train_acc", "val_loss", "val_acc", "top1", "top5", "spike_rate", "lr"]
    
    for key in expected_keys:
        if key not in history:
            history[key] = []
        elif not isinstance(history[key], list):
            history[key] = [history[key]]

    target_length = 0
    if "epoch" in history and history["epoch"]:
        target_length = len(history["epoch"])
    elif "train_loss" in history and history["train_loss"]:
        target_length = len(history["train_loss"])
    elif "val_loss" in history and history["val_loss"]:
        target_length = len(history["val_loss"])

    if target_length > 0:
        for key in expected_keys:
            current_len = len(history[key])
            if current_len < target_length:
                history[key].extend([np.nan] * (target_length - current_len))
            elif current_len > target_length:
                history[key] = history[key][:target_length]
            
        if "lr" in history and all(pd.isna(x) for x in history["lr"]):
             del history["lr"]

# --- ZMODYFIKOWANO: Połączona krzywa straty i dokładności (loss_accuracy_curves.png) ---
if history and "epoch" in history and len(history["epoch"]) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(18, 6), constrained_layout=True)

    # Wykres straty
    if "train_loss" in history and "val_loss" in history:
        axes[0].plot(history["epoch"], history["train_loss"], label="Strata treningowa")
        axes[0].plot(history["epoch"], history["val_loss"], label="Strata walidacyjna")
        axes[0].set_xlabel("Epoka")
        axes[0].set_ylabel("Strata")
        axes[0].set_title("Krzywa straty treningowej i walidacyjnej")
        axes[0].legend()
        axes[0].grid(True)
    else:
        axes[0].set_title("Brak danych o stracie")

    # Wykres dokładności (bez val_acc)
    if "train_acc" in history: 
        axes[1].plot(history["epoch"], [acc * 100 for acc in history["train_acc"]], label="Dokładność treningowa")
        if "top1" in history and history["top1"] and len(history["top1"]) == len(history["epoch"]):
            axes[1].plot(history["epoch"], [acc * 100 for acc in history["top1"]], label="Dokładność Top-1")
        if "top5" in history and history["top5"] and len(history["top5"]) == len(history["epoch"]):
            axes[1].plot(history["epoch"], [acc * 100 for acc in history["top5"]], label="Dokładność Top-5")
        axes[1].set_xlabel("Epoka")
        axes[1].set_ylabel("Dokładność (%)")
        axes[1].set_title("Krzywa dokładności treningowej, Top-1 i Top-5")
        axes[1].legend()
        axes[1].grid(True)
    else:
        axes[1].set_title("Brak danych o dokładności")

    plt.savefig(PLOTS_DIR / "loss_accuracy_curves.png")
    plt.close()
    print(f"Wygenerowano {PLOTS_DIR / 'loss_accuracy_curves.png'}")

# --- Średnia częstość wyładowań (mean_firing_rate.png) ---
if history and "spike_rate" in history and "epoch" in history and len(history["epoch"]) > 0:
    if history["spike_rate"] and len(history["spike_rate"]) > 0:
        if isinstance(history["spike_rate"][0], list):
            num_layers = len(history["spike_rate"][0])
            firing_rates_per_layer = [[] for _ in range(num_layers)]
            
            for epoch_idx, epoch_rates in enumerate(history["spike_rate"]):
                if isinstance(epoch_rates, list) and len(epoch_rates) == num_layers:
                    for i in range(num_layers):
                        firing_rates_per_layer[i].append(epoch_rates[i])
                else:
                    for i in range(num_layers):
                        firing_rates_per_layer[i].append(np.nan)
            
            plt.figure(figsize=(12, 7), constrained_layout=True)
            for i, layer_rates in enumerate(firing_rates_per_layer):
                if not all(np.isnan(rate) for rate in layer_rates):
                    plt.plot(history["epoch"][:len(layer_rates)], layer_rates, label=f"Warstwa LIF {i+1} Częstość Wyładowań")
            plt.xlabel("Epoka")
            plt.ylabel("Średnia częstość wyładowań")
            plt.title("Średnia częstość wyładowań na warstwę LIF w trakcie epok")
            plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
            plt.grid(True)
            plt.savefig(PLOTS_DIR / "mean_firing_rate.png")
            plt.close()
            print(f"Wygenerowano {PLOTS_DIR / 'mean_firing_rate.png'}")

# --- Metryki CSV (metrics.csv) ---
if history:
    df_history = pd.DataFrame(history)
    if "spike_rate" in df_history.columns and not df_history["spike_rate"].empty:
        if isinstance(df_history["spike_rate"].iloc[0], list):
            max_layers = 0
            for sr_list in df_history["spike_rate"]:
                if isinstance(sr_list, list):
                    max_layers = max(max_layers, len(sr_list))

            for i in range(max_layers):
                df_history[f"firing_rate_layer_{i+1}"] = df_history["spike_rate"].apply(lambda x: x[i] if isinstance(x, list) and len(x) > i else np.nan)
    
    if "spike_rate" in history and "spike_rate" in df_history.columns and isinstance(history["spike_rate"][0], list):
        avg_firing_rate_per_epoch = [np.mean(sr_list) if isinstance(sr_list, list) and sr_list else np.nan for sr_list in history["spike_rate"]]
        df_history["avg_firing_rate"] = pd.Series(avg_firing_rate_per_epoch, index=df_history.index)
    elif "spike_rate" in df_history.columns and not df_history["spike_rate"].empty:
        if not isinstance(df_history["spike_rate"].iloc[0], list):
            df_history["avg_firing_rate"] = df_history["spike_rate"]

    if "spike_rate" in df_history.columns and "firing_rate_layer_1" in df_history.columns:
        df_history = df_history.drop(columns=["spike_rate"])

    df_history.to_csv(PLOTS_DIR / "metrics.csv", index=False)
    print(f"Wygenerowano {PLOTS_DIR / 'metrics.csv'}")

# --- Ewaluacja najlepszego modelu ---
all_labels, all_probs, all_preds = None, None, None
val_acc_final, top1_final, top5_final = 0.0, 0.0, 0.0
final_precision_weighted, final_recall_weighted, final_f1_weighted = 0.0, 0.0, 0.0
final_precision_macro, final_recall_macro, final_f1_macro = 0.0, 0.0, 0.0
classification_report_text = "Brak danych ewaluacyjnych."

try:
    # --- POPRAWKA: Inicjalizacja modelu ResNet18 SNN ---
    model = resnet18(**MODEL_PARAMS).to(device)
    criterion = nn.CrossEntropyLoss()
except Exception as e:
    print(f"Błąd inicjalizacji modelu ResNet18: {e}")
    model = None

if model is not None and BEST_MODEL_PATH.exists():
    print(f"Wczytywanie najlepszego modelu z {BEST_MODEL_PATH} do ewaluacji...")
    try:
        # --- ZMIANA: load state dict na odpowiednie urządzenie ---
        model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
        model.eval()

        if not TEST_ROOT.exists():
            print(f"\nOSTRZEŻENIE: Ścieżka '{TEST_ROOT}' nie istnieje lokalnie.")
        else:
            test_ds = EventDatasetBIN(TEST_ROOT, CLASSES, T=DATASET_T, H=180, W=240) 
            if len(test_ds) == 0:
                print(f"\nOSTRZEŻENIE: Brak danych testowych.")
            else:
                # --- ZMIANA: Dostosowanie num_workers i pin_memory dla DataLoader ---
                num_workers_eval = 4 if device.type == "cuda" else 0
                pin_memory_eval = (device.type == "cuda")
                test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=num_workers_eval, pin_memory=pin_memory_eval)
                
                print("Ewaluacja najlepszego modelu na zbiorze testowym...")
                val_loss, val_acc_final, top1_final, top5_final, all_labels, all_probs, _ = evaluate(
                    model, test_loader, device, criterion, CLASSES
                )
                all_preds = np.argmax(all_probs, axis=1)
                
                final_precision_weighted, final_recall_weighted, final_f1_weighted, _ = precision_recall_fscore_support(
                    all_labels, all_preds, average='weighted', zero_division=0
                )
                final_precision_macro, final_recall_macro, final_f1_macro, _ = precision_recall_fscore_support(
                    all_labels, all_preds, average='macro', zero_division=0
                )

                print(f"Dokładność: {val_acc_final*100:.2f}% | Top-1: {top1_final*100:.2f}% | Top-5: {top5_final*100:.2f}%")
                print(f"Precyzja (weighted): {final_precision_weighted*100:.2f}% | (macro): {final_precision_macro*100:.2f}%")
                print(f"Recall (weighted): {final_recall_weighted*100:.2f}% | (macro): {final_recall_macro*100:.2f}%")
                print(f"F1-Score (weighted): {final_f1_weighted*100:.2f}% | (macro): {final_f1_macro*100:.2f}%")
                
                classification_report_text = classification_report(all_labels, all_preds, target_names=CLASSES, zero_division=0)
    except Exception as e:
        print(f"Błąd podczas ewaluacji najlepszego modelu: {e}")
else:
    print("Błąd: Model nie zainicjalizowany lub brak pliku best_model.pth.")

# --- 5. Macierz Pomyłek ---
if NUM_CLASSES <= 20 and all_labels is not None and all_preds is not None and len(CLASSES) > 1:
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(12, 10), constrained_layout=True)
    sns.heatmap(cm, annot=False, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES)
    plt.xlabel("Przewidziana etykieta")
    plt.ylabel("Prawdziwa etykieta")
    plt.title("Macierz Pomyłek")
    plt.savefig(PLOTS_DIR / "confusion_matrix.png")
    plt.close()
    print(f"Wygenerowano {PLOTS_DIR / 'confusion_matrix.png'}")
else:
    print(f"Ostrzeżenie: Pomijam macierz pomyłek (dla {NUM_CLASSES} klas jest za duża lub brak danych ewaluacyjnych).")

# --- 6. Raport Klasyfikacji (classification_report.txt) ---
if all_labels is not None and all_preds is not None and len(CLASSES) > 0:
    with open(PLOTS_DIR / "classification_report.txt", "w", encoding="utf-8") as f:
        f.write("Raport Klasyfikacji:\n")
        f.write(classification_report_text)
    print(f"Wygenerowano {PLOTS_DIR / 'classification_report.txt'}")

# --- Rozkład częstości wyładowań w ostatniej epoce (Bar plot) ---
if history and "spike_rate" in history and "epoch" in history and len(history["epoch"]) > 0:
    if isinstance(history["spike_rate"][-1], list):
        final_firing_rates = history["spike_rate"][-1]
        layer_names = [f"LIF {i+1}" for i in range(len(final_firing_rates))]

        plt.figure(figsize=(10, 6), constrained_layout=True)
        sns.barplot(x=layer_names, y=final_firing_rates, palette="viridis")
        plt.xlabel("Warstwa LIF")
        plt.ylabel("Średnia częstość wyładowań")
        plt.title("Średnia częstość wyładowań na warstwę LIF (ostatnia epoka)")
        plt.xticks(rotation=45, ha='right')
        plt.savefig(PLOTS_DIR / "final_firing_rate_distribution.png")
        plt.close()
        print(f"Wygenerowano {PLOTS_DIR / 'final_firing_rate_distribution.png'}")

# --- Dokładność per klasa (Bar plot) ---
if all_labels is not None and all_preds is not None and len(CLASSES) > 0:
    per_class_accuracy = {}
    for i, class_name in enumerate(CLASSES):
        class_indices = np.where(all_labels == i)[0]
        if len(class_indices) > 0:
            class_true = all_labels[class_indices]
            class_pred = all_preds[class_indices]
            per_class_accuracy[class_name] = accuracy_score(class_true, class_pred)
        else:
            per_class_accuracy[class_name] = 0.0

    class_names_sorted = sorted(per_class_accuracy.keys())
    accuracies_sorted = [per_class_accuracy[name] for name in class_names_sorted]

    plt.figure(figsize=(15, 8), constrained_layout=True)
    sns.barplot(x=class_names_sorted, y=[acc * 100 for acc in accuracies_sorted], palette="magma")
    plt.xlabel("Klasa")
    plt.ylabel("Dokładność (%)")
    plt.title("Dokładność per klasa na zbiorze testowym")
    plt.xticks(rotation=90, fontsize=8)
    plt.savefig(PLOTS_DIR / "per_class_accuracy.png")
    plt.close()
    print(f"Wygenerowano {PLOTS_DIR / 'per_class_accuracy.png'}")

# --- Nowe wykresy korelacji i trendów ---
val_accs_for_plots, avg_firing_rates_for_plots, epochs_for_plots = [], [], []
correlation = np.nan

if history and "val_acc" in history and "spike_rate" in history and "epoch" in history:
    for i, epoch_rates in enumerate(history["spike_rate"]):
        if isinstance(epoch_rates, list) and epoch_rates and i < len(history["val_acc"]):
            avg_firing_rates_for_plots.append(np.mean(epoch_rates))
            val_accs_for_plots.append(history["val_acc"][i])
            epochs_for_plots.append(history["epoch"][i])

    valid_data_points = [(sr, va, ep) for sr, va, ep in zip(avg_firing_rates_for_plots, val_accs_for_plots, epochs_for_plots) if not np.isnan(sr) and not np.isnan(va)]
    if valid_data_points:
        avg_firing_rates_for_plots = [item[0] for item in valid_data_points]
        val_accs_for_plots = [item[1] for item in valid_data_points]
        epochs_for_plots = [item[2] for item in valid_data_points]

# Wykres trendów w czasie
if val_accs_for_plots and avg_firing_rates_for_plots and len(epochs_for_plots) > 0:
    fig, ax1 = plt.subplots(figsize=(14, 7), constrained_layout=True)

    color = 'tab:blue'
    ax1.set_xlabel("Epoka")
    ax1.set_ylabel("Dokładność Walidacyjna (%)", color=color)
    ax1.plot(epochs_for_plots, [acc * 100 for acc in val_accs_for_plots], color=color, marker='o', linestyle='-')
    ax1.tick_params(axis='y', labelcolor=color)
    ax1.grid(True)
    ax1.set_ylim(0, 100)

    ax2 = ax1.twinx()
    color = 'tab:red'
    ax2.set_ylabel("Średnia Częstość Wyładowań (%)", color=color)
    ax2.plot(epochs_for_plots, [sr * 100 for sr in avg_firing_rates_for_plots], color=color, marker='x', linestyle='--')
    ax2.tick_params(axis='y', labelcolor=color)

    fig.suptitle("Średnia częstość wyładowań i dokładność walidacyjna w funkcji epok", fontsize=16)
    plt.savefig(PLOTS_DIR / "firing_rate_vs_val_acc_over_epochs.png")
    plt.close()
    print(f"Wygenerowano {PLOTS_DIR / 'firing_rate_vs_val_acc_over_epochs.png'}")

    # Scatter plot bez legendy
    plt.figure(figsize=(10, 6), constrained_layout=True)
    sns.scatterplot(x=[sr * 100 for sr in avg_firing_rates_for_plots], y=[acc * 100 for acc in val_accs_for_plots],
                    hue=epochs_for_plots, palette="viridis", legend=False,
                    size=epochs_for_plots, sizes=(50, 200), alpha=0.7)
    plt.xlabel("Średnia częstość wyładowań (%)")
    plt.ylabel("Dokładność Walidacyjna (%)")
    plt.title("Zależność: Średnia Częstość Wyładowań vs. Dokładność Walidacyjna (z epoką)")
    plt.grid(True)
    plt.ylim(0, 100)
    plt.savefig(PLOTS_DIR / "avg_firing_rate_vs_val_acc_scatter.png")
    plt.close()
    print(f"Wygenerowano {PLOTS_DIR / 'avg_firing_rate_vs_val_acc_scatter.png'}")

# Wykres korelacji
if len(val_accs_for_plots) > 1 and len(avg_firing_rates_for_plots) > 1:
    correlation, _ = pearsonr(avg_firing_rates_for_plots, val_accs_for_plots)
    print(f"Korelacja Pearsona (Avg Częstość Wyładowań vs. Val Accuracy): {correlation:.4f}")

    plt.figure(figsize=(10, 6), constrained_layout=True)
    sns.regplot(x=np.array(avg_firing_rates_for_plots) * 100, y=np.array(val_accs_for_plots) * 100,
                scatter_kws={'alpha':0.6}, line_kws={'color':'red'})
    plt.xlabel("Średnia częstość wyładowań (%)")
    plt.ylabel("Dokładność Walidacyjna (%)")
    plt.title(f"Korelacja: Średnia Częstość Wyładowań vs. Dokładność Walidacyjna (Pearson r = {correlation:.2f})")
    plt.grid(True)
    plt.ylim(0, 100)
    plt.savefig(PLOTS_DIR / "correlation_firing_rate_val_acc.png")
    plt.close()
    print(f"Wygenerowano {PLOTS_DIR / 'correlation_firing_rate_val_acc.png'}")

# --- 7. Generowanie raportu Markdown ---
report_content = f"""
# Raport z Eksperymentu - Sieć Impulsowa (SNN) ResNet18

**Data i czas generowania raportu:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
**Lokalizacja wyników:** {RESULTS_DIR}
**Lokalizacja wizualizacji:** {PLOTS_DIR}

---

## Przegląd Modelu i Treningu
*   **Model:** ResNet18 SNN
*   **Zbiór danych:** Synthetic DVS (format H5)
*   **Liczba klas:** {NUM_CLASSES}
*   **Kroki czasowe (Time Steps, T):** {DATASET_T}
*   **Urządzenie:** {device} (wykorzystano GPU, jeśli dostępne; w przeciwnym razie CPU)

**Parametry sieci (LIFNode):**
*   Próg (`thresh`): {MODEL_PARAMS['thresh']} | Stała czasowa (`tau`): {MODEL_PARAMS['tau']} | Parametr surrogatowy (`gamma`): {MODEL_PARAMS['gamma']}
*   Dropout (`dropout_p`): {MODEL_PARAMS['dropout_p']}

---

## Wyniki Ewaluacji na Zbiorze Testowym
"""
if all_labels is not None and all_preds is not None and len(CLASSES) > 0:
    report_content += f"""
### Ogólne Metryki
*   **Najlepsza Dokładność Walidacyjna (z treningu):** {history["val_acc"][np.argmax(history["val_acc"])]*100:.2f}% (epoka {history["epoch"][np.argmax(history["val_acc"])] if "val_acc" in history and history["val_acc"] else 'N/A'})
*   **Dokładność na Zbiorze Testowym (Accuracy):** {val_acc_final*100:.2f}%
*   **Top-1 Accuracy na Zbiorze Testowym:** {top1_final*100:.2f}%
*   **Top-5 Accuracy na Zbiorze Testowym:** {top5_final*100:.2f}%
*   **Precyzja (Weighted Average):** {final_precision_weighted*100:.2f}% | (Macro Average): {final_precision_macro*100:.2f}%
*   **Recall (Weighted Average):** {final_recall_weighted*100:.2f}% | (Macro Average): {final_recall_macro*100:.2f}%
*   **F1-Score (Weighted Average):** {final_f1_weighted*100:.2f}% | (Macro Average): {final_f1_macro*100:.2f}%

"""

Używane urządzenie: cpu
Wykryta liczba klas: 101
Wczytywanie checkpointu z C:\Users\Kacper\OneDrive\Desktop\SNN\results\results-real_ResNet\checkpoint.pth...
Historia wczytana pomyślnie.
Wygenerowano C:\Users\Kacper\OneDrive\Desktop\SNN\plots\results-real_ResNet\loss_accuracy_curves.png
Wygenerowano C:\Users\Kacper\OneDrive\Desktop\SNN\plots\results-real_ResNet\mean_firing_rate.png
Wygenerowano C:\Users\Kacper\OneDrive\Desktop\SNN\plots\results-real_ResNet\metrics.csv
Wczytywanie najlepszego modelu z C:\Users\Kacper\OneDrive\Desktop\SNN\results\results-real_ResNet\best_model.pth do ewaluacji...


C:\Users\Kacper\AppData\Local\Temp\ipykernel_12096\3258530038.py:531: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(BEST_MODEL_PATH, map_loc

Błąd podczas ewaluacji najlepszego modelu: name 'EventDatasetH5' is not defined
Ostrzeżenie: Pomijam macierz pomyłek (dla 101 klas jest za duża lub brak danych ewaluacyjnych).
Wygenerowano C:\Users\Kacper\OneDrive\Desktop\SNN\plots\results-real_ResNet\final_firing_rate_distribution.png
Wygenerowano C:\Users\Kacper\OneDrive\Desktop\SNN\plots\results-real_ResNet\firing_rate_vs_val_acc_over_epochs.png
Wygenerowano C:\Users\Kacper\OneDrive\Desktop\SNN\plots\results-real_ResNet\avg_firing_rate_vs_val_acc_scatter.png
Korelacja Pearsona (Avg Częstość Wyładowań vs. Val Accuracy): -0.9836
Wygenerowano C:\Users\Kacper\OneDrive\Desktop\SNN\plots\results-real_ResNet\correlation_firing_rate_val_acc.png


In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import snntorch as snn
from snntorch import surrogate

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from pathlib import Path
from tqdm import tqdm
from collections import Counter
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix, top_k_accuracy_score, accuracy_score, precision_recall_fscore_support
import h5py # Nadal importowane, choć dla BIN nieużywane w datasecie
import math
from scipy.stats import pearsonr # For correlation

# --- FUNKCJE I KLASY DLA DANYCH SYNTETYCZNYCH (.H5) ---
def load_h5_events(path):
    with h5py.File(path, "r") as f:
        data = f["events"][:]

    t = data[:, 0].astype(np.float32)
    x = data[:, 1].astype(np.int32)
    y = data[:, 2].astype(np.int32)
    p = np.where(data[:, 3] > 0, 1, -1)

    return x, y, t, p


class EventDatasetH5(Dataset):
    def __init__(self, root_dir, classes, T=10, H=180, W=240,
                 clip_value=5.0, augment=False):
        self.root_dir = Path(root_dir)
        self.classes = classes
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        self.T, self.H, self.W = T, H, W
        self.clip_value = clip_value
        self.augment = augment

        self.samples = []
        
        if not self.root_dir.exists():
            print(f"Warning: Dataset root directory does not exist: {self.root_dir}")
            return # Exit early if root_dir doesn't exist

        for cls in classes:
            class_dir = self.root_dir / cls
            if not class_dir.exists():
                print(f"Ostrzeżenie: Katalog klasy {class_dir} nie istnieje. Pomijam.")
                continue
            for f in class_dir.glob("*.h5"): # Szukamy plików .h5
                self.samples.append((f, self.class_to_idx[cls]))
        
        if not self.samples:
            print(f"Warning: No event files found in {self.root_dir} for given classes (looking for .h5).")


    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        x, y, t, p = load_h5_events(path)

        if len(t) == 0:
            # Zwróć tensor zer o odpowiednim kształcie i etykietę, jeśli brak zdarzeń
            return torch.zeros((2, self.H, self.W, self.T), dtype=torch.float32), torch.tensor(label, dtype=torch.long)

        if self.augment and np.random.rand() > 0.5:
            x = (self.W - 1) - x

        # Handle cases where t.max() - t.min() is 0 (single event or all events at same time)
        t_min = t.min()
        t_max = t.max()
        if (t_max - t_min) < 1e-9: # Avoid division by zero
            t_normalized = np.zeros_like(t) # All events in the first time bin
        else:
            t_normalized = (t - t_min) / (t_max - t_min)
            
        voxels = np.zeros((2, self.H, self.W, self.T), dtype=np.float32)
        time_bins = (t_normalized * (self.T - 1)).astype(np.int32)

        for xi, yi, pi, bi in zip(x, y, p, time_bins):
            if 0 <= xi < self.W and 0 <= yi < self.H:
                ch = 0 if pi > 0 else 1
                voxels[ch, yi, xi, bi] += 1.0

        voxels = np.clip(voxels, 0.0, self.clip_value)
        voxels /= self.clip_value

        return torch.from_numpy(voxels), torch.tensor(label, dtype=torch.long)

# --- KONIEC FUNKCJI I KLAS DLA DANYCH SYNTETYCZNYCH ---


class SeqToANNContainer(nn.Module):
    def __init__(self, *args):
        super().__init__()
        # Poprawka: args powinien być traktowany jako lista modułów dla Sequential
        if len(args) == 1 and isinstance(args[0], nn.Module):
            self.module = args[0]
        else:
            self.module = nn.Sequential(*args)
    def forward(self, x_seq: torch.Tensor):
        y_shape = [x_seq.shape[0], x_seq.shape[1]] # Poprawka: x_seq.shape[0] dla batch size, x_seq.shape[1] dla time steps
        y_seq = self.module(x_seq.flatten(0, 1).contiguous())
        y_shape.extend(y_seq.shape[1:])
        return y_seq.view(y_shape)

class SpikeModule(nn.Module):
    def __init__(self, module):
        super().__init__()
        self.ann_module = module
    def forward(self, x):
        B, T, *spatial_dims = x.shape
        out = self.ann_module(x.reshape(B * T, *spatial_dims))
        BT, *spatial_dims = out.shape
        out = out.view(B, T, *spatial_dims).contiguous()
        return out

def fire_function(gamma):
    class ZIF(torch.autograd.Function):
        @staticmethod
        def forward(ctx, input):
            out = (input >= 0).float()
            ctx.save_for_backward(input)
            return out
        @staticmethod
        def backward(ctx, grad_output):
            (input, ) = ctx.saved_tensors
            grad_input = grad_output.clone()
            tmp = (input.abs() < gamma/2).float() / gamma
            grad_input = grad_input * tmp
            return grad_input, None
    return ZIF.apply

class LIFSpike(nn.Module):
    def __init__(self, thresh=0.5, tau=0.25, gamma=1.0):
        super(LIFSpike, self).__init__()
        self.thresh = thresh
        self.tau = tau
        self.gamma = gamma
        self.fire_fn = fire_function(self.gamma)
        self.total_firing_events = 0 # Zmieniono nazwę zmiennej
        self.num_neurons = 0
        self.num_time_steps = 0
        self.batch_size = 0
    def forward(self, x):
        mem = torch.zeros_like(x[:, 0])
        spikes = []
        T = x.shape[1] # Poprawka: Czas to drugi wymiar (batch, time, channels, H, W)
        self.total_firing_events = 0 # Zmieniono nazwę zmiennej
        self.num_neurons = x.shape[2:].numel()
        self.num_time_steps = T
        self.batch_size = x.shape[0] # Poprawka: Batch size to pierwszy wymiar
        for t in range(T):
            mem = mem * self.tau + x[:, t, ...]
            spike = self.fire_fn(mem - self.thresh)
            mem = (1 - spike) * mem
            spikes.append(spike)
            self.total_firing_events += spike.sum().item() # Zmieniono nazwę zmiennej
        return torch.stack(spikes, dim=1)
    def get_avg_firing_rate(self):
        if self.batch_size * self.num_time_steps * self.num_neurons == 0:
            return 0.0
        return self.total_firing_events / (self.batch_size * self.num_time_steps * self.num_neurons) # Zmieniono nazwę zmiennej

def add_dimention(x, T):
    x.unsqueeze_(1)
    x = x.repeat(1, T, 1, 1, 1)
    return x

class tdBatchNorm(nn.BatchNorm2d):
    def __init__(self, channel):
        super(tdBatchNorm, self).__init__(channel)
        self.weight.data.mul_(0.5)
    def forward(self, x):
        B, T, *spatial_dims = x.shape
        out = super().forward(x.reshape(B * T, *spatial_dims))
        BT, *spatial_dims = out.shape
        out = out.view(B, T, *spatial_dims).contiguous()
        return out

class VGG(nn.Module):
    ''' VGG model '''
    def __init__(self, cfg, num_classes=101, batch_norm=True, in_c=2, dropout_p=0.4, **lif_parameters):
        super(VGG, self).__init__()
        self.T = lif_parameters.get('T', 10)
        self.dropout_p = dropout_p
        self.lif_modules = nn.ModuleList()
        self.features, out_c = make_layers(cfg, batch_norm, in_c, dropout_p=self.dropout_p, lif_modules=self.lif_modules, **lif_parameters)
        self.avgpool = SeqToANNContainer(nn.AdaptiveAvgPool2d((1, 1)))
        self.classifier = nn.Sequential(
            SeqToANNContainer(nn.Linear(out_c, num_classes)),
        )
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                # Poprawka: kernel_size jest tuplem (width, height), trzeba użyć elementów
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
                if m.bias is not None:
                    m.bias.data.zero_()
    def get_avg_firing_rates(self): # Zmieniono z get_avg_spike_rates
        rates = []
        for lif_module in self.lif_modules:
            rates.append(lif_module.get_avg_firing_rate())
        return rates
    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 2)
        x = self.classifier(x)
        return x

def make_layers(cfg, batch_norm=False, in_c=2, dropout_p=0.4, lif_modules=None, **lif_parameters):
    layers = []
    in_channels = in_c
    num_lif_layers = 0
    for v in cfg:
        if v != 'M':
            num_lif_layers += 1
    current_lif_count = 0
    for v_idx, v_val in enumerate(cfg):
        if v_val == 'M':
            layers += [SpikeModule(nn.AvgPool2d(kernel_size=2, stride=2))]
        else:
            # Type casting to ensure integer, preventing TypeError
            try:
                in_channels_int = int(in_channels)
                v_val_int = int(v_val)
            except ValueError:
                raise ValueError(f"Expected integer for channel count, but got '{in_channels}' or '{v_val}' at layer {v_idx} in VGG config.")

            conv2d = SpikeModule(nn.Conv2d(in_channels_int, v_val_int, kernel_size=3, padding=1))
            lif = LIFSpike(**lif_parameters)
            if lif_modules is not None:
                lif_modules.append(lif)
            if batch_norm:
                bn = tdBatchNorm(v_val_int)
                layers += [conv2d, bn, lif]
            else:
                layers += [conv2d, lif]
            current_lif_count += 1
            if current_lif_count < num_lif_layers:
                layers += [SpikeModule(nn.Dropout2d(p=dropout_p))]
            in_channels = v_val_int # Update in_channels for the next layer

    return nn.Sequential(*layers), in_channels

cfg = {
    'A': [64, 'M', 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M'], # VGG11
    'B': [64, 64, 'M', 128, 128, 'M', 256, 256, 'M', 512, 512, 512, 512], # VGG13
    'D': [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'M', 512, 512, 512, 512, 512, 512], # VGG16
}

def vgg11(*args, **kwargs):
    """VGG 11-layer model (configuration "A") with batch normalization"""
    return VGG(cfg['A'], *args, **kwargs)

def vgg13(*args, **kwargs):
    """VGG 13-layer model (configuration "B") with batch normalization"""
    return VGG(cfg['B'], *args, **kwargs)

def vgg16(*args, **kwargs):
    """VGG 16-layer model (configuration "D") with batch normalization"""
    return VGG(cfg['D'], *args, **kwargs)

def evaluate(model, loader, device, criterion, classes):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    all_labels = []
    all_probs = []
    # Dodano pasek postępu do ewaluacji
    with torch.no_grad():
        for x, y in tqdm(loader, desc="Ewaluacja modelu"): # Dodano tqdm
            x, y = x.to(device), y.to(device)
            x = x.permute(0, 4, 1, 2, 3) # Z (B,C,H,W,T) na (B,T,C,H,W)
            out_seq = model(x)
            out = out_seq.mean(1)
            loss = criterion(out, y)
            total_loss += loss.item()
            probs = F.softmax(out, dim=1)
            preds = probs.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
            all_labels.extend(y.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    val_loss = total_loss / len(loader)
    val_acc = correct / total
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    top1 = top_k_accuracy_score(all_labels, all_probs, k=1)
    top5 = top_k_accuracy_score(all_labels, all_probs, k=min(5, len(classes)))
    return val_loss, val_acc, top1, top5, all_labels, all_probs, 0.0

# --- Konfiguracja dla wizualizacji ---
# --- ZMIANA: Ponowne włączenie detekcji GPU ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Używane urządzenie: {device}")

# --- ZMIANA: Ponowne włączenie ustawień CUDA backend ---
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

# Ścieżki bazowe dla wyników i wykresów
BASE_SNN_DIR = Path(r"C:\Users\Kacper\OneDrive\Desktop\SNN")
EXPERIMENT_NAME = "results-synthetic_101-ResNet" # Zmieniona nazwa eksperymentu na ResNet18 dla danych syntetycznych

RESULTS_DIR = BASE_SNN_DIR / "results" / EXPERIMENT_NAME
PLOTS_DIR = BASE_SNN_DIR / "plots" / EXPERIMENT_NAME
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = RESULTS_DIR / "checkpoint.pth"
BEST_MODEL_PATH = RESULTS_DIR / "best_model.pth"

# Ścieżki do danych syntetycznych H5
TRAIN_ROOT = BASE_SNN_DIR / "data" / "Synthetic_DVS-full-split" / "train"
TEST_ROOT  = BASE_SNN_DIR / "data" / "Synthetic_DVS-full-split" / "test"

# Wczytanie nazw klas
CLASSES = []
NUM_CLASSES = 0
try:
    if TRAIN_ROOT.exists():
        CLASSES = sorted([p.name for p in TRAIN_ROOT.iterdir() if p.is_dir()])
        try:
            CLASSES = sorted(CLASSES, key=int)
        except ValueError:
            pass
        NUM_CLASSES = len(CLASSES)
        if NUM_CLASSES == 0:
            print(f"Ostrzeżenie: Nie znaleziono katalogów klas w '{TRAIN_ROOT}'. Użycie domyślnych 101 klas.")
            NUM_CLASSES = 101
            CLASSES = [f"{i}" for i in range(NUM_CLASSES)]
    else:
        print(f"Ostrzeżenie: Katalog danych '{TRAIN_ROOT}' nie istnieje. Użycie domyślnych 101 klas.")
        NUM_CLASSES = 101
        CLASSES = [f"Class_{i}" for i in range(NUM_CLASSES)]
except Exception as e:
    print(f"Błąd podczas wczytywania klas: {e}. Użycie domyślnych 101 klas.")
    NUM_CLASSES = 101
    CLASSES = [f"Class_{i}" for i in range(NUM_CLASSES)]

print(f"Wykryta liczba klas: {NUM_CLASSES}")

# Hiperparametry modelu ResNet18 SNN
DATASET_T = 10 # Zgodnie z twoim kodem treningowym dla syntetycznych
MODEL_PARAMS = {
    "num_classes": NUM_CLASSES,
    "in_channels": 2, # Zmieniono z in_c na in_channels dla ResNet
    "thresh": 0.7, # Przykład z kodu treningowego ResNet
    "tau": 0.95,   # Przykład z kodu treningowego ResNet
    "gamma": 1.0,  # Dodano gamma dla LIFNode
    "dropout_p": 0.1, # Przykład z kodu treningowego ResNet
    "T": DATASET_T # T jest przekazywane do ResNet_SNN, a stamtąd do LIFNode
}

# --- Wczytanie historii z checkpointu ---
history = None
if CHECKPOINT_PATH.exists():
    print(f"Wczytywanie checkpointu z {CHECKPOINT_PATH}...")
    try:
        # --- ZMIANA: load checkpoint na odpowiednie urządzenie ---
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
        history = checkpoint["history"]
        print("Historia wczytana pomyślnie.")
    except Exception as e:
        print(f"Błąd podczas wczytywania checkpointu: {e}")
        history = None
else:
    print(f"Błąd: Plik checkpointu nie znaleziony pod {CHECKPOINT_PATH}.")

# Robustne przetwarzanie historii
if history:
    expected_keys = ["epoch", "train_loss", "train_acc", "val_loss", "val_acc", "top1", "top5", "spike_rate", "lr"]
    
    for key in expected_keys:
        if key not in history:
            history[key] = []
        elif not isinstance(history[key], list):
            history[key] = [history[key]]

    target_length = 0
    if "epoch" in history and history["epoch"]:
        target_length = len(history["epoch"])
    elif "train_loss" in history and history["train_loss"]:
        target_length = len(history["train_loss"])
    elif "val_loss" in history and history["val_loss"]:
        target_length = len(history["val_loss"])

    if target_length > 0:
        for key in expected_keys:
            current_len = len(history[key])
            if current_len < target_length:
                history[key].extend([np.nan] * (target_length - current_len))
            elif current_len > target_length:
                history[key] = history[key][:target_length]
            
        if "lr" in history and all(pd.isna(x) for x in history["lr"]):
             del history["lr"]

# --- ZMODYFIKOWANO: Połączona krzywa straty i dokładności (loss_accuracy_curves.png) ---
if history and "epoch" in history and len(history["epoch"]) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(18, 6), constrained_layout=True)

    # Wykres straty
    if "train_loss" in history and "val_loss" in history:
        axes[0].plot(history["epoch"], history["train_loss"], label="Strata treningowa")
        axes[0].plot(history["epoch"], history["val_loss"], label="Strata walidacyjna")
        axes[0].set_xlabel("Epoka")
        axes[0].set_ylabel("Strata")
        axes[0].set_title("Krzywa straty treningowej i walidacyjnej")
        axes[0].legend()
        axes[0].grid(True)
    else:
        axes[0].set_title("Brak danych o stracie")

    # Wykres dokładności (bez val_acc)
    if "train_acc" in history: 
        axes[1].plot(history["epoch"], [acc * 100 for acc in history["train_acc"]], label="Dokładność treningowa")
        if "top1" in history and history["top1"] and len(history["top1"]) == len(history["epoch"]):
            axes[1].plot(history["epoch"], [acc * 100 for acc in history["top1"]], label="Dokładność Top-1")
        if "top5" in history and history["top5"] and len(history["top5"]) == len(history["epoch"]):
            axes[1].plot(history["epoch"], [acc * 100 for acc in history["top5"]], label="Dokładność Top-5")
        axes[1].set_xlabel("Epoka")
        axes[1].set_ylabel("Dokładność (%)")
        axes[1].set_title("Krzywa dokładności treningowej, Top-1 i Top-5")
        axes[1].legend()
        axes[1].grid(True)
    else:
        axes[1].set_title("Brak danych o dokładności")

    plt.savefig(PLOTS_DIR / "loss_accuracy_curves.png")
    plt.close()
    print(f"Wygenerowano {PLOTS_DIR / 'loss_accuracy_curves.png'}")

# --- Średnia częstość wyładowań (mean_firing_rate.png) ---
if history and "spike_rate" in history and "epoch" in history and len(history["epoch"]) > 0:
    if history["spike_rate"] and len(history["spike_rate"]) > 0:
        if isinstance(history["spike_rate"][0], list):
            num_layers = len(history["spike_rate"][0])
            firing_rates_per_layer = [[] for _ in range(num_layers)]
            
            for epoch_idx, epoch_rates in enumerate(history["spike_rate"]):
                if isinstance(epoch_rates, list) and len(epoch_rates) == num_layers:
                    for i in range(num_layers):
                        firing_rates_per_layer[i].append(epoch_rates[i])
                else:
                    for i in range(num_layers):
                        firing_rates_per_layer[i].append(np.nan)
            
            plt.figure(figsize=(12, 7), constrained_layout=True)
            for i, layer_rates in enumerate(firing_rates_per_layer):
                if not all(np.isnan(rate) for rate in layer_rates):
                    plt.plot(history["epoch"][:len(layer_rates)], layer_rates, label=f"Warstwa LIF {i+1} Częstość Wyładowań")
            plt.xlabel("Epoka")
            plt.ylabel("Średnia częstość wyładowań")
            plt.title("Średnia częstość wyładowań na warstwę LIF w trakcie epok")
            plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
            plt.grid(True)
            plt.savefig(PLOTS_DIR / "mean_firing_rate.png")
            plt.close()
            print(f"Wygenerowano {PLOTS_DIR / 'mean_firing_rate.png'}")

# --- Metryki CSV (metrics.csv) ---
if history:
    df_history = pd.DataFrame(history)
    if "spike_rate" in df_history.columns and not df_history["spike_rate"].empty:
        if isinstance(df_history["spike_rate"].iloc[0], list):
            max_layers = 0
            for sr_list in df_history["spike_rate"]:
                if isinstance(sr_list, list):
                    max_layers = max(max_layers, len(sr_list))

            for i in range(max_layers):
                df_history[f"firing_rate_layer_{i+1}"] = df_history["spike_rate"].apply(lambda x: x[i] if isinstance(x, list) and len(x) > i else np.nan)
    
    if "spike_rate" in history and "spike_rate" in df_history.columns and isinstance(history["spike_rate"][0], list):
        avg_firing_rate_per_epoch = [np.mean(sr_list) if isinstance(sr_list, list) and sr_list else np.nan for sr_list in history["spike_rate"]]
        df_history["avg_firing_rate"] = pd.Series(avg_firing_rate_per_epoch, index=df_history.index)
    elif "spike_rate" in df_history.columns and not df_history["spike_rate"].empty:
        if not isinstance(df_history["spike_rate"].iloc[0], list):
            df_history["avg_firing_rate"] = df_history["spike_rate"]

    if "spike_rate" in df_history.columns and "firing_rate_layer_1" in df_history.columns:
        df_history = df_history.drop(columns=["spike_rate"])

    df_history.to_csv(PLOTS_DIR / "metrics.csv", index=False)
    print(f"Wygenerowano {PLOTS_DIR / 'metrics.csv'}")

# --- Ewaluacja najlepszego modelu ---
all_labels, all_probs, all_preds = None, None, None
val_acc_final, top1_final, top5_final = 0.0, 0.0, 0.0
final_precision_weighted, final_recall_weighted, final_f1_weighted = 0.0, 0.0, 0.0
final_precision_macro, final_recall_macro, final_f1_macro = 0.0, 0.0, 0.0
classification_report_text = "Brak danych ewaluacyjnych."

try:
    # --- POPRAWKA: Inicjalizacja modelu ResNet18 SNN ---
    model = resnet18(**MODEL_PARAMS).to(device)
    criterion = nn.CrossEntropyLoss()
except Exception as e:
    print(f"Błąd inicjalizacji modelu ResNet18: {e}")
    model = None

if model is not None and BEST_MODEL_PATH.exists():
    print(f"Wczytywanie najlepszego modelu z {BEST_MODEL_PATH} do ewaluacji...")
    try:
        # --- ZMIANA: load state dict na odpowiednie urządzenie ---
        model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
        model.eval()

        if not TEST_ROOT.exists():
            print(f"\nOSTRZEŻENIE: Ścieżka '{TEST_ROOT}' nie istnieje lokalnie.")
        else:
            test_ds = EventDatasetH5(TEST_ROOT, CLASSES, T=DATASET_T, H=180, W=240) 
            if len(test_ds) == 0:
                print(f"\nOSTRZEŻENIE: Brak danych testowych.")
            else:
                # --- ZMIANA: Dostosowanie num_workers i pin_memory dla DataLoader ---
                num_workers_eval = 4 if device.type == "cuda" else 0
                pin_memory_eval = (device.type == "cuda")
                test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=num_workers_eval, pin_memory=pin_memory_eval)
                
                print("Ewaluacja najlepszego modelu na zbiorze testowym...")
                val_loss, val_acc_final, top1_final, top5_final, all_labels, all_probs, _ = evaluate(
                    model, test_loader, device, criterion, CLASSES
                )
                all_preds = np.argmax(all_probs, axis=1)
                
                final_precision_weighted, final_recall_weighted, final_f1_weighted, _ = precision_recall_fscore_support(
                    all_labels, all_preds, average='weighted', zero_division=0
                )
                final_precision_macro, final_recall_macro, final_f1_macro, _ = precision_recall_fscore_support(
                    all_labels, all_preds, average='macro', zero_division=0
                )

                print(f"Dokładność: {val_acc_final*100:.2f}% | Top-1: {top1_final*100:.2f}% | Top-5: {top5_final*100:.2f}%")
                print(f"Precyzja (weighted): {final_precision_weighted*100:.2f}% | (macro): {final_precision_macro*100:.2f}%")
                print(f"Recall (weighted): {final_recall_weighted*100:.2f}% | (macro): {final_recall_macro*100:.2f}%")
                print(f"F1-Score (weighted): {final_f1_weighted*100:.2f}% | (macro): {final_f1_macro*100:.2f}%")
                
                classification_report_text = classification_report(all_labels, all_preds, target_names=CLASSES, zero_division=0)
    except Exception as e:
        print(f"Błąd podczas ewaluacji najlepszego modelu: {e}")
else:
    print("Błąd: Model nie zainicjalizowany lub brak pliku best_model.pth.")

# --- 5. Macierz Pomyłek ---
if NUM_CLASSES <= 20 and all_labels is not None and all_preds is not None and len(CLASSES) > 1:
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(12, 10), constrained_layout=True)
    sns.heatmap(cm, annot=False, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES)
    plt.xlabel("Przewidziana etykieta")
    plt.ylabel("Prawdziwa etykieta")
    plt.title("Macierz Pomyłek")
    plt.savefig(PLOTS_DIR / "confusion_matrix.png")
    plt.close()
    print(f"Wygenerowano {PLOTS_DIR / 'confusion_matrix.png'}")
else:
    print(f"Ostrzeżenie: Pomijam macierz pomyłek (dla {NUM_CLASSES} klas jest za duża lub brak danych ewaluacyjnych).")

# --- 6. Raport Klasyfikacji (classification_report.txt) ---
if all_labels is not None and all_preds is not None and len(CLASSES) > 0:
    with open(PLOTS_DIR / "classification_report.txt", "w", encoding="utf-8") as f:
        f.write("Raport Klasyfikacji:\n")
        f.write(classification_report_text)
    print(f"Wygenerowano {PLOTS_DIR / 'classification_report.txt'}")

# --- Rozkład częstości wyładowań w ostatniej epoce (Bar plot) ---
if history and "spike_rate" in history and "epoch" in history and len(history["epoch"]) > 0:
    if isinstance(history["spike_rate"][-1], list):
        final_firing_rates = history["spike_rate"][-1]
        layer_names = [f"LIF {i+1}" for i in range(len(final_firing_rates))]

        plt.figure(figsize=(10, 6), constrained_layout=True)
        sns.barplot(x=layer_names, y=final_firing_rates, palette="viridis")
        plt.xlabel("Warstwa LIF")
        plt.ylabel("Średnia częstość wyładowań")
        plt.title("Średnia częstość wyładowań na warstwę LIF (ostatnia epoka)")
        plt.xticks(rotation=45, ha='right')
        plt.savefig(PLOTS_DIR / "final_firing_rate_distribution.png")
        plt.close()
        print(f"Wygenerowano {PLOTS_DIR / 'final_firing_rate_distribution.png'}")

# --- Dokładność per klasa (Bar plot) ---
if all_labels is not None and all_preds is not None and len(CLASSES) > 0:
    per_class_accuracy = {}
    for i, class_name in enumerate(CLASSES):
        class_indices = np.where(all_labels == i)[0]
        if len(class_indices) > 0:
            class_true = all_labels[class_indices]
            class_pred = all_preds[class_indices]
            per_class_accuracy[class_name] = accuracy_score(class_true, class_pred)
        else:
            per_class_accuracy[class_name] = 0.0

    class_names_sorted = sorted(per_class_accuracy.keys())
    accuracies_sorted = [per_class_accuracy[name] for name in class_names_sorted]

    plt.figure(figsize=(15, 8), constrained_layout=True)
    sns.barplot(x=class_names_sorted, y=[acc * 100 for acc in accuracies_sorted], palette="magma")
    plt.xlabel("Klasa")
    plt.ylabel("Dokładność (%)")
    plt.title("Dokładność per klasa na zbiorze testowym")
    plt.xticks(rotation=90, fontsize=8)
    plt.savefig(PLOTS_DIR / "per_class_accuracy.png")
    plt.close()
    print(f"Wygenerowano {PLOTS_DIR / 'per_class_accuracy.png'}")

# --- Nowe wykresy korelacji i trendów ---
val_accs_for_plots, avg_firing_rates_for_plots, epochs_for_plots = [], [], []
correlation = np.nan

if history and "val_acc" in history and "spike_rate" in history and "epoch" in history:
    for i, epoch_rates in enumerate(history["spike_rate"]):
        if isinstance(epoch_rates, list) and epoch_rates and i < len(history["val_acc"]):
            avg_firing_rates_for_plots.append(np.mean(epoch_rates))
            val_accs_for_plots.append(history["val_acc"][i])
            epochs_for_plots.append(history["epoch"][i])

    valid_data_points = [(sr, va, ep) for sr, va, ep in zip(avg_firing_rates_for_plots, val_accs_for_plots, epochs_for_plots) if not np.isnan(sr) and not np.isnan(va)]
    if valid_data_points:
        avg_firing_rates_for_plots = [item[0] for item in valid_data_points]
        val_accs_for_plots = [item[1] for item in valid_data_points]
        epochs_for_plots = [item[2] for item in valid_data_points]

# Wykres trendów w czasie
if val_accs_for_plots and avg_firing_rates_for_plots and len(epochs_for_plots) > 0:
    fig, ax1 = plt.subplots(figsize=(14, 7), constrained_layout=True)

    color = 'tab:blue'
    ax1.set_xlabel("Epoka")
    ax1.set_ylabel("Dokładność Walidacyjna (%)", color=color)
    ax1.plot(epochs_for_plots, [acc * 100 for acc in val_accs_for_plots], color=color, marker='o', linestyle='-')
    ax1.tick_params(axis='y', labelcolor=color)
    ax1.grid(True)
    ax1.set_ylim(0, 100)

    ax2 = ax1.twinx()
    color = 'tab:red'
    ax2.set_ylabel("Średnia Częstość Wyładowań (%)", color=color)
    ax2.plot(epochs_for_plots, [sr * 100 for sr in avg_firing_rates_for_plots], color=color, marker='x', linestyle='--')
    ax2.tick_params(axis='y', labelcolor=color)

    fig.suptitle("Średnia częstość wyładowań i dokładność walidacyjna w funkcji epok", fontsize=16)
    plt.savefig(PLOTS_DIR / "firing_rate_vs_val_acc_over_epochs.png")
    plt.close()
    print(f"Wygenerowano {PLOTS_DIR / 'firing_rate_vs_val_acc_over_epochs.png'}")

    # Scatter plot bez legendy
    plt.figure(figsize=(10, 6), constrained_layout=True)
    sns.scatterplot(x=[sr * 100 for sr in avg_firing_rates_for_plots], y=[acc * 100 for acc in val_accs_for_plots],
                    hue=epochs_for_plots, palette="viridis", legend=False,
                    size=epochs_for_plots, sizes=(50, 200), alpha=0.7)
    plt.xlabel("Średnia częstość wyładowań (%)")
    plt.ylabel("Dokładność Walidacyjna (%)")
    plt.title("Zależność: Średnia Częstość Wyładowań vs. Dokładność Walidacyjna (z epoką)")
    plt.grid(True)
    plt.ylim(0, 100)
    plt.savefig(PLOTS_DIR / "avg_firing_rate_vs_val_acc_scatter.png")
    plt.close()
    print(f"Wygenerowano {PLOTS_DIR / 'avg_firing_rate_vs_val_acc_scatter.png'}")

# Wykres korelacji
if len(val_accs_for_plots) > 1 and len(avg_firing_rates_for_plots) > 1:
    correlation, _ = pearsonr(avg_firing_rates_for_plots, val_accs_for_plots)
    print(f"Korelacja Pearsona (Avg Częstość Wyładowań vs. Val Accuracy): {correlation:.4f}")

    plt.figure(figsize=(10, 6), constrained_layout=True)
    sns.regplot(x=np.array(avg_firing_rates_for_plots) * 100, y=np.array(val_accs_for_plots) * 100,
                scatter_kws={'alpha':0.6}, line_kws={'color':'red'})
    plt.xlabel("Średnia częstość wyładowań (%)")
    plt.ylabel("Dokładność Walidacyjna (%)")
    plt.title(f"Korelacja: Średnia Częstość Wyładowań vs. Dokładność Walidacyjna (Pearson r = {correlation:.2f})")
    plt.grid(True)
    plt.ylim(0, 100)
    plt.savefig(PLOTS_DIR / "correlation_firing_rate_val_acc.png")
    plt.close()
    print(f"Wygenerowano {PLOTS_DIR / 'correlation_firing_rate_val_acc.png'}")

# --- 7. Generowanie raportu Markdown ---
report_content = f"""
# Raport z Eksperymentu - Sieć Impulsowa (SNN) ResNet18

**Data i czas generowania raportu:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
**Lokalizacja wyników:** {RESULTS_DIR}
**Lokalizacja wizualizacji:** {PLOTS_DIR}

---

## Przegląd Modelu i Treningu
*   **Model:** ResNet18 SNN
*   **Zbiór danych:** Synthetic DVS (format H5)
*   **Liczba klas:** {NUM_CLASSES}
*   **Kroki czasowe (Time Steps, T):** {DATASET_T}
*   **Urządzenie:** {device} (wykorzystano GPU, jeśli dostępne; w przeciwnym razie CPU)

**Parametry sieci (LIFNode):**
*   Próg (`thresh`): {MODEL_PARAMS['thresh']} | Stała czasowa (`tau`): {MODEL_PARAMS['tau']} | Parametr surrogatowy (`gamma`): {MODEL_PARAMS['gamma']}
*   Dropout (`dropout_p`): {MODEL_PARAMS['dropout_p']}

---

## Wyniki Ewaluacji na Zbiorze Testowym
"""
if all_labels is not None and all_preds is not None and len(CLASSES) > 0:
    report_content += f"""
### Ogólne Metryki
*   **Najlepsza Dokładność Walidacyjna (z treningu):** {history["val_acc"][np.argmax(history["val_acc"])]*100:.2f}% (epoka {history["epoch"][np.argmax(history["val_acc"])] if "val_acc" in history and history["val_acc"] else 'N/A'})
*   **Dokładność na Zbiorze Testowym (Accuracy):** {val_acc_final*100:.2f}%
*   **Top-1 Accuracy na Zbiorze Testowym:** {top1_final*100:.2f}%
*   **Top-5 Accuracy na Zbiorze Testowym:** {top5_final*100:.2f}%
*   **Precyzja (Weighted Average):** {final_precision_weighted*100:.2f}% | (Macro Average): {final_precision_macro*100:.2f}%
*   **Recall (Weighted Average):** {final_recall_weighted*100:.2f}% | (Macro Average): {final_recall_macro*100:.2f}%
*   **F1-Score (Weighted Average):** {final_f1_weighted*100:.2f}% | (Macro Average): {final_f1_macro*100:.2f}%

"""

Używane urządzenie: cpu
Wykryta liczba klas: 101
Wczytywanie checkpointu z C:\Users\Kacper\OneDrive\Desktop\SNN\results\results-synthetic_101-ResNet\checkpoint.pth...
Historia wczytana pomyślnie.
Wygenerowano C:\Users\Kacper\OneDrive\Desktop\SNN\plots\results-synthetic_101-ResNet\loss_accuracy_curves.png
Wygenerowano C:\Users\Kacper\OneDrive\Desktop\SNN\plots\results-synthetic_101-ResNet\mean_firing_rate.png
Wygenerowano C:\Users\Kacper\OneDrive\Desktop\SNN\plots\results-synthetic_101-ResNet\metrics.csv
Błąd inicjalizacji modelu ResNet18: name 'resnet18' is not defined
Błąd: Model nie zainicjalizowany lub brak pliku best_model.pth.
Ostrzeżenie: Pomijam macierz pomyłek (dla 101 klas jest za duża lub brak danych ewaluacyjnych).


C:\Users\Kacper\AppData\Local\Temp\ipykernel_7724\3033785558.py:587: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=layer_names, y=final_firing_rates, palette="viridis")


Wygenerowano C:\Users\Kacper\OneDrive\Desktop\SNN\plots\results-synthetic_101-ResNet\final_firing_rate_distribution.png
Wygenerowano C:\Users\Kacper\OneDrive\Desktop\SNN\plots\results-synthetic_101-ResNet\firing_rate_vs_val_acc_over_epochs.png
Wygenerowano C:\Users\Kacper\OneDrive\Desktop\SNN\plots\results-synthetic_101-ResNet\avg_firing_rate_vs_val_acc_scatter.png
Korelacja Pearsona (Avg Częstość Wyładowań vs. Val Accuracy): -0.9899
Wygenerowano C:\Users\Kacper\OneDrive\Desktop\SNN\plots\results-synthetic_101-ResNet\correlation_firing_rate_val_acc.png
